# VWAP skew / strangle — with fallback

Four-regime short-options strategy. Capital is always invested.

1. **spot ≥ VWAP + 20 bps** → skew: short PUT far (−3) + CALL near (+1), pyramided 1–2–3–4.
2. **spot ≤ VWAP − 20 bps** → mirror skew: CALL far (+3) + PUT near (−1).
3. **range < 15 bps** → tight strangle bracketing spot, all 10/side at once.
4. **else** → wide ±4 strangle (park the cash) until a better regime fires.

**Risk exits (all regimes):** profit-take at 25% of credit · fixed stop at 40% loss · trailing (arms at 15%, give-back 15%) · time backstop / regime yield.


In [ ]:
from engine import BirdsEye
from strategies.vwap_skew_strangle import VwapSkewStrangle

be = BirdsEye(
    data_dir          = "/mnt/INTERNAL_DATA/INTERN_DATA/OPRA_DATA_1SEC/0DTE/SPY",
    strategy_cls      = VwapSkewStrangle,
    index             = "SPY",
    lot_size          = 100,
    starting_cash     = 1_000_000.0,
    margin_per_lot    = 10_000.0,
    fields            = ("spot", "atm_strike",
                         "ce_bid_0", "ce_ask_0", "pe_bid_0", "pe_ask_0",
                         "volume"),
    strategy_kwargs   = {
        "lots":             10,
        "dev_bps_thresh":   20.0,
        "range_bps_max":    15.0,
        "range_win":        900,
        "pyramid_schedule": (1, 2, 3, 4),
        "group_interval":   10 * 60,
        "skew_max_hold":    2 * 60 * 60,
        "tight_hold":       2 * 60 * 60,
        # --- risk exits ---
        "profit_take_frac": 0.25,
        "stop_loss_frac":   0.40,
        "trail_arm_frac":   0.15,
        "trail_frac":       0.15,
        # "profit_target": 500.0, "stop_loss": 400.0, "trail_stop": 250.0,
        # --- VWAP ---
        "vwap_use_volume":      True,
        "volume_field":         "volume",
        "volume_is_cumulative": False,
        "quote_persist":        60,
    },
    cost_kwargs       = {"txn_cost_per_lot": 0.85},
    n_workers         = 40,
    collect_perseclog = True,
    # days=["20240102", "20240104"],
)
res = be.run()


In [ ]:
import matplotlib.pyplot as plt

DAY = res.days[0]

print("=== per-day summary ===")
display(res.summary)


In [ ]:
def display_stats(stats: dict):
    from IPython.display import display, HTML

    def fmt(k, v):
        if isinstance(v, float):
            if "pct" in k or "cagr" in k or "calmar" in k:
                color = "green" if v > 0 else "red"
                return f'<span style="color:{color}">{v:+.2f}%</span>'
            if "pnl" in k or "day" in k or "win" in k or "loss" in k or "cost" in k:
                color = "green" if v > 0 else ("red" if v < 0 else "inherit")
                return f'<span style="color:{color}">${v:+,.2f}</span>'
            return f"{v:.4f}"
        if isinstance(v, int):
            return f"{v:,}"
        return str(v)

    rows = "".join(
        f"<tr><td style='padding:4px 16px 4px 0;color:gray;font-size:13px'>{k}</td>"
        f"<td style='padding:4px 0;font-size:13px;font-weight:500'>{fmt(k,v)}</td></tr>"
        for k, v in stats.items()
    )
    display(HTML(f"<table style='border-collapse:collapse'>{rows}</table>"))

display_stats(res.stats())


In [ ]:
led = res.tradelog
print(f"=== trade ledger: {len(led)} fills ===")
display(led.head(10))
print("\nfills by signal:")
display(led.groupby("signal")["exe_cost"].agg(["count","sum"]).round(2))


In [ ]:
sl = res.perseclog(DAY)
print(f"=== per-second log {DAY}: {len(sl)} rows ===")
display(sl.iloc[2000:2005])

import matplotlib.pyplot as plt
sl.plot(y=["spot","vwap"], figsize=(11,3), title=f"{DAY} — spot vs VWAP"); plt.show()
sl.plot(y=["dev_bps","range_bps"], figsize=(11,3), title=f"{DAY} — regime alphas (bps)"); plt.show()


In [ ]:
from engine.plots import plot_equity
plot_equity(res)


In [ ]:
# ============================================================================
# PnL statistics by TRADE TYPE (one episode = one round trip)
#   skew  : above_vwap / below_vwap  +  skew_add  +  stop (profit_take/stop_loss/trailing/skew_maxhold)
#   tight : calm_range  +  stop (profit_take/stop_loss/trailing) or tight_done
#   wide  : open_fallback  +  a stop (profit_take/stop_loss/trailing) or wide_to_up/dn/calm
# mid-p PnL uses ONLY fill prices (SELL = +price, BUY = -price) x lots x lot_size.
# t-cost is the ledger's exe_cost (txn + brokerage + spread).
# ============================================================================
import numpy as np
import pandas as pd

_OPENERS = {"above_vwap": "skew", "below_vwap": "skew", "calm_range": "tight", "open_fallback": "wide"}
_ADDERS  = {"skew_add": "skew"}
_CLOSERS = {"profit_take": "skew", "stop_loss": "skew", "trailing": "skew",
            "skew_maxhold": "skew", "tight_done": "tight",
            "wide_to_up": "wide", "wide_to_dn": "wide", "wide_to_calm": "wide"}


def _segment(df):
    df   = df.reset_index(drop=True)
    sigs = df["signal"].tolist()
    runs, start = [], 0
    for k in range(1, len(df) + 1):
        if k == len(df) or sigs[k] != sigs[start]:
            runs.append(df.iloc[start:k]); start = k

    eps, cur = [], None
    for run in runs:
        sig = run["signal"].iloc[0]
        if sig in _OPENERS:
            if cur is not None: eps.append(cur)
            cur = {"type": _OPENERS[sig], "fills": [run]}
        elif cur is not None:
            cur["fills"].append(run)
            if sig in _CLOSERS or sig == "eod_square_off":
                eps.append(cur); cur = None
        else:
            t = _ADDERS.get(sig) or _CLOSERS.get(sig) or "unknown"
            cur = {"type": t, "fills": [run]}
            if sig in _CLOSERS or sig == "eod_square_off":
                eps.append(cur); cur = None
    if cur is not None: eps.append(cur)
    for e in eps: e["fills"] = pd.concat(e["fills"], ignore_index=True)
    return eps


def _episode_stats(ep, lot_size):
    f      = ep["fills"]
    sign   = np.where(f["action"] == "SELL", 1.0, -1.0)
    midp   = float((sign * f["fill_price"] * f["lots"] * lot_size).sum())
    tcost  = float(f["exe_cost"].sum())
    openl  = float(f.loc[f["action"] == "SELL", "lots"].sum())
    net_pos= float(np.where(f["action"] == "BUY", f["lots"], -f["lots"]).sum())
    return {"type": ep["type"], "n_fills": len(f), "midp_pnl": midp, "tcost": tcost,
            "net_pnl": midp - tcost, "open_lots": openl, "flat": abs(net_pos) < 1e-9}


def trade_pnl_stats(led, lot_size, verbose=True):
    """Per-trade-type PnL stats. Returns (per_type_table, per_trade_table)."""
    if led is None or len(led) == 0:
        if verbose: print("empty ledger"); return pd.DataFrame(), pd.DataFrame()
    eps = []
    for _, g in led.groupby("day", sort=True):
        eps += _segment(g)
    per_trade = pd.DataFrame([_episode_stats(e, lot_size) for e in eps])
    order = ["skew", "tight", "wide"]
    agg = per_trade.groupby("type").agg(
        trades=("type","size"), midp_pnl=("midp_pnl","sum"),
        tcost=("tcost","sum"), net_pnl=("net_pnl","sum"), open_lots=("open_lots","sum"))
    agg = agg.reindex([t for t in order if t in agg.index]
                      + [t for t in agg.index if t not in order])
    agg["midp_per_lot"]   = agg["midp_pnl"] / agg["open_lots"].replace(0, np.nan)
    agg["midp_per_trade"] = agg["midp_pnl"] / agg["trades"]
    agg["tcost_per_trade"]= agg["tcost"]    / agg["trades"]
    total = pd.Series({
        "trades": agg["trades"].sum(), "midp_pnl": agg["midp_pnl"].sum(),
        "tcost": agg["tcost"].sum(), "net_pnl": agg["net_pnl"].sum(),
        "open_lots": agg["open_lots"].sum(),
        "midp_per_lot":    agg["midp_pnl"].sum() / max(agg["open_lots"].sum(), 1e-9),
        "midp_per_trade":  agg["midp_pnl"].sum() / max(agg["trades"].sum(), 1),
        "tcost_per_trade": agg["tcost"].sum()    / max(agg["trades"].sum(), 1),
    }, name="TOTAL")
    table = pd.concat([agg, total.to_frame().T])
    if verbose:
        show = table.copy()
        for c in ["midp_pnl","tcost","net_pnl","midp_per_lot","midp_per_trade","tcost_per_trade"]:
            show[c] = show[c].astype(float).round(2)
        show["trades"]    = show["trades"].astype(int)
        show["open_lots"] = show["open_lots"].astype(float).round(0).astype(int)
        print("=== PnL by trade type (mid-price) ==="); print(show.to_string())
        n_open = int((~per_trade["flat"]).sum())
        if n_open: print(f"\nNote: {n_open} episode(s) not flat.")
        print(f"\nTOTAL NET PnL (after costs):     {table.loc['TOTAL','net_pnl']:,.2f}")
        print(f"TOTAL mid-p PnL (before costs):  {table.loc['TOTAL','midp_pnl']:,.2f}")
        print(f"TOTAL t-costs:                   {table.loc['TOTAL','tcost']:,.2f}")
        print(f"mid-p PnL per opening lot:       {table.loc['TOTAL','midp_per_lot']:,.4f}")
    return table, per_trade


pnl_by_type, pnl_by_trade = trade_pnl_stats(res.tradelog, lot_size=res.cfg.lot_size)
